finite diff on a uniform
mesh choice improvements (chebyshev
how it doesn't work for classical bvp discretization
explain fornberg
Matrix representation
BVP with fornbeg
compare all errors


# The Finite Difference Method we're used to  
Given an ordinary differential equation, we discretize it via a finite difference approximation of the derivative and represent it as a matrix problem.

Using the Taylor Series Expansion of u(x+h) and u(x-h)  
where we assume the matrix is uniform and where u(x) = $u_i$, u(x-h) = $u_{i-1}$, and u(x+h) = $u_{i+1}$  
### Finite Difference for approximations of derivatives

$u'(t) = \frac{u_{i+1} - u_{i-1}}{2h} + O(h^2)$  

$u''(t) = \frac{u_{i-1} +2u_i + u_{i+1}}{h^2} + O(h^2)$  

Then we proceed with the normal numerical scheme, however,  
what if we use a non-uniform grid where the distance between points isn't h  

### Non-uniform grid choice
Chebyshev nodes of the second kind:
$x_i = cos(\frac{i \pi}{N})$ for i=0,1,...,N  
If n = 10
$x_0 = 1.0$  
$x_1 = 0.95$  
$x_2 = 0.81$  
$x_3 = 0.58$  
...  

So now if we try to find the central finite difference at $x_2$ for example, we see the step size to the left is different than that to the right  
$h_1 = |x_2 - x_1 | =  0.14$  
$h_2 = | x_2 - x_3 | = 0.23$  
$u(x-h_1)$ and $u(x+h_2)$  
and hence instead of simply doing the taylor series expansion of $u(x+h) - u(x-h)$ to cancel out $u''$,  
it becomes a problem of solving a linear system to determine the weights  

### Fornberg Algorithm

### Example

### Use on a non-uniform Boundary Value Problem

In [35]:
# chebyshev nodes of second kind
a,b = -1,1
N=10
function chebyshev_nodes(a,b,N)
    X = []
    for i in 0:N
        x = cos(i*π/N)
        push!(X,x)
        # println(x)
    end
    return X
end
chebyshev_nodes(-1,1,4)
# calculate_weights(1,3chebyshev_nodes(a,b,N))


5-element Vector{Any}:
  1.0
  0.7071067811865476
  6.123233995736766e-17
 -0.7071067811865475
 -1.0

In [ ]:
# fornberg (if we don't want to use the one below)

In [ ]:
# BVP using chebyshev nodes and fornberg


# What happens when the grid isn't uniform?
a

In [49]:
#############################################################
# Fornberg algorithm

# This implements the Fornberg (1988) algorithm (https://doi.org/10.1090/S0025-5718-1988-0935077-0)
# to obtain Finite Difference weights over arbitrary points to arbitrary order.

function calculate_weights(order::Int, x0::T, x::AbstractVector) where {T <: Real}
    #=
        order: The derivative order for which we need the coefficients
        x0   : The point in the array 'x' for which we need the coefficients
        x    : A dummy array with relative coordinates, e.g., central differences
               need coordinates centred at 0 while those at boundaries need
               coordinates starting from 0 to the end point

        The approximation order of the stencil is automatically determined from
        the number of requested stencil points.
    =#
    N = length(x)
    @assert order<N "Not enough points for the requested order."
    M = order
    c1 = one(T)
    c4 = x[1] - x0
    C = zeros(T, N, M + 1)
    C[1, 1] = 1
    @inbounds for i in 1:(N - 1)
        i1 = i + 1
        mn = min(i, M)
        c2 = one(T)
        c5 = c4
        c4 = x[i1] - x0
        for j in 0:(i - 1)
            j1 = j + 1
            c3 = x[i1] - x[j1]
            c2 *= c3
            if j == i - 1
                for s in mn:-1:1
                    s1 = s + 1
                    C[i1, s1] = c1 * (s * C[i, s] - c5 * C[i, s1]) / c2
                end
                C[i1, 1] = -c1 * c5 * C[i, 1] / c2
            end
            for s in mn:-1:1
                s1 = s + 1
                C[j1, s1] = (c4 * C[j1, s1] - s * C[j1, s]) / c3
            end
            C[j1, 1] = c4 * C[j1, 1] / c3
        end
        c1 = c2
    end
    #=
        This is to fix the problem of numerical instability which occurs when the sum of the stencil_coefficients is not
        exactly 0.
        https://scicomp.stackexchange.com/questions/11249/numerical-derivative-and-finite-difference-coefficients-any-update-of-the-fornb
        Stack Overflow answer on this issue.
        http://epubs.siam.org/doi/pdf/10.1137/S0036144596322507 - Modified Fornberg Algorithm
    =#
    _C = C[:, end]
    if order != 0
        _C[div(N, 2) + 1] -= sum(_C)
    end
    display(C)
    return _C
end

X = chebyshev_nodes(-1,1,4)
calculate_weights(Int(2),X[3],X)

5×3 Matrix{Float64}:
  0.0  -0.5          -1.0
 -0.0   1.41421       4.0
  1.0  -4.44089e-16  -6.0
  0.0  -1.41421       4.0
 -0.0   0.5          -1.0

5-element Vector{Float64}:
 -1.0000000000000002
  4.000000000000001
 -6.0
  3.9999999999999987
 -0.9999999999999998

In [44]:
using LinearAlgebra

X = chebyshev_nodes(-1,1,20)
calculate_weights(Int(1),X[3],X)

# Compute weights and approximate the derivative
weights = calculate_weights(1, X[3], X)
f_values = sin.(X)
approx = dot(weights, f_values)
exact = cos(X[3])

println(weights)

println("x0 (3rd Chebyshev node): $X[3]")
println()
println("Approximate f'(x0):  $approx")
println("Exact f'(x0):        $exact")
println("Absolute error:      $(abs(approx - exact))")

[-10.21586454726538, 27.298667732483953, -4.9797965697654565, -16.652791531124983, 7.040294042680412, -4.099205106965183, 2.7527638409423503, -2.011805206337479, 1.5575365158350527, -1.2584599161585448, 1.05146222423827, -0.9029418901400228, 0.7936044933348396, -0.7117199556938546, 0.6498393924658127, -0.603076911374608, 0.5681580876808239, -0.5428695859059794, 0.5257311121191327, -0.5157976287833848, 0.256271407734227]
x0 (3rd Chebyshev node): Any[1.0, 0.9876883405951378, 0.9510565162951535, 0.8910065241883679, 0.8090169943749475, 0.7071067811865476, 0.5877852522924731, 0.45399049973954686, 0.30901699437494745, 0.15643446504023092, 6.123233995736766e-17, -0.1564344650402306, -0.30901699437494734, -0.45399049973954675, -0.587785252292473, -0.7071067811865475, -0.8090169943749473, -0.8910065241883678, -0.9510565162951535, -0.9876883405951377, -1.0][3]

Approximate f'(x0):  0.5808233782431589
Exact f'(x0):        0.5808233782431591
Absolute error:      2.220446049250313e-16


In [40]:
# Derivative tests for Fornberg (1998) Lagrange polynomial derivatives
# Bryan Kaiser
# 12/20/15

using DataArrays
using PyPlot
using PyCall
@pyimport numpy as np
@pyimport pylab as py


## ============================================================================
# function declaration

function weights(z,x,n,m)
# From Bengt Fornbergs (1998) SIAM Review paper.
#  	Input Parameters
#	z location where approximations are to be accurate,
#	x(0:nd) grid point locations, found in x(0:n)
#	n one less than total number of grid points; n must
#	not exceed the parameter nd below,
#	nd dimension of x- and c-arrays in calling program
#	x(0:nd) and c(0:nd,0:m), respectively,
#	m highest derivative for which weights are sought,
#	Output Parameter
#	c(0:nd,0:m) weights at grid locations x(0:n) for derivatives
#	of order 0:m, found in c(0:n,0:m)
#      	dimension x(0:nd),c(0:nd,0:m)

	c = zeros(n+1,m+1);
	c1 = 1.0;
	c4 = x[1+0]-z;
	for k=0:m
  		for j=0:n
    			c[1+j,1+k] = 0.0;
  		end
	end
	c[1+0,1+0] = 1.0;
	for  i=1:n
  		mn = min(i,m);
  		c2 = 1.0;
  		c5 = c4;
  		c4 = x[1+i]-z;
  		for j=0:i-1
    			c3 = x[1+i]-x[1+j];
    			c2 = c2*c3;
    			if (j == i-1)
      				for k=mn:-1:1
        				c[1+i,1+k] = c1*(k*c[1+i-1,1+k-1]-c5*c[1+i-1,1+k])/c2;
			      	end
      			c[1+i,1+0] = -c1*c5*c[1+i-1,1+0]/c2;
    			end
    			for k=mn:-1:1
      				c[1+j,1+k] = (c4*c[1+j,1+k]-k*c[1+j,1+k-1])/c3;
    			end
    			c[1+j,1+0] = c4*c[1+j,1+0]/c3;
  		end
  		c1 = c2;
	end
   	return c
end # weights function


## ============================================================================
# domain parameters

const Lx = 3000.0 # km, domain size
const Ly = Lx # km
const Lxcenter = 0.0 # x value @ the center of the grid
const Lycenter = 0.0 # y value @ the center of the grid
const N = 2^20 # series length (must be at least even)
const dx = Lx/float(N) # km, uniform longitudinal grid spacing
const dy = Ly/float(N) # km, uniform longitudinal grid spacing
x = collect(0.5*dx:dx:dx*N)-(Lx/2.0-Lxcenter) # km, centered uniform grid 


## ============================================================================
# signal

k = 2.0*pi/(Lx/5.0)
s = sin(x.*k)
dsdx = cos(x.*k).*k
d2sdx2 = -sin(x.*k).*k^2

figure(1)
plot(x,s,"b",x,dsdx,"r",x,d2sdx2,"g")
xlabel("x")
title("signal")
show()


##=============================================================================
# compute the uniform grid weights

c = weights(x[3],x[1:5],4,2)'; # 5 point stencil weights
dsF = zeros(1,N); # derivative of the input function
m = 1 # order of the derivative
   
dsF = zeros(Float64,N,1)
for j = 3:N-2      
	jm2 = j-2;
        jp2 = j+2;
        dsF[j,1] = vecdot(c[m+1,:],s[jm2:jp2]);
end
error = abs(dsdx-dsF)

@show(N)
figure(2)
semilogy(x[3:N-2],error[3:N-2],"k")
xlabel("x")
title("Lagrange polynomial 1st derivative error")
show()

LoadError: ArgumentError: Package PyPlot not found in current path.
- Run `import Pkg; Pkg.add("PyPlot")` to install the PyPlot package.

In [19]:
import Pkg; Pkg.add("DataArrays")

    Updating registry at `C:\Users\Brady\.julia\registries\General.toml`
   Resolving package versions...
   Installed JpegTurbo_jll ───────── v3.1.5+0
   Installed Libmount_jll ────────── v2.42.0+0
   Installed GR_jll ──────────────── v0.73.24+0
   Installed Opus_jll ────────────── v1.6.1+0
   Installed GR ──────────────────── v0.73.24
   Installed LERC_jll ────────────── v4.1.0+0
   Installed ConcurrentUtilities ─── v2.5.1
   Installed Preferences ─────────── v1.5.2
   Installed DataArrays ──────────── v0.7.0
   Installed Cairo_jll ───────────── v1.18.6+0
   Installed Xorg_libXinerama_jll ── v1.1.7+0
   Installed PlotUtils ───────────── v1.2.0
   Installed Missings ────────────── v0.4.5
   Installed OpenSSL ─────────────── v1.6.1
   Installed libva_jll ───────────── v2.23.0+0
   Installed Xorg_libxkbfile_jll ─── v1.2.0+0
   Installed Xorg_libpciaccess_jll ─ v0.18.1+0
   Installed xkbcommon_jll ───────── v1.13.0+0
   Installed Pango_jll ───────────── v1.57.1+0
   Installed XZ_jll ────